# Differential Equations — Session 25
## Section 5.3: Nonlinear Models

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives

Students should be able to:

1. formulate a nonlinear spring model;
2. distinguish hardening and softening restoring forces;
3. use potential energy to interpret nonlinear motion;
4. derive the nonlinear pendulum equation;
5. assess the small-angle linearization;
6. distinguish oscillation from rotation using pendulum energy;
7. derive the catenary equation and solution;
8. formulate inverse-square rocket motion and escape velocity;
9. explain why variable mass requires momentum balance.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Core 90-minute path

| Time | Topic |
|---:|---|
| 0–25 min | Nonlinear springs and potential energy |
| 25–55 min | Nonlinear pendulum and linearization |
| 55–70 min | Catenary model |
| 70–84 min | Rocket motion and escape velocity |
| 84–90 min | Variable-mass caution and exit check |

The chain-lifting model is an optional extension.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp, quad
from scipy.optimize import brentq
from IPython.display import display, Markdown

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True
np.set_printoptions(precision=6, suppress=True)

def solve_second_order(accel, t_span, y0, points=1000, **kwargs):
    def rhs(t, z):
        x, v = z
        return [v, accel(t, x, v)]
    t_eval = np.linspace(t_span[0], t_span[1], points)
    return solve_ivp(rhs, t_span, y0, t_eval=t_eval, **kwargs)

print('Notebook ready.')
print('Interactive widgets available:', WIDGETS_AVAILABLE)

## Formal theory reference

### Definition 5.3-A — Nonlinear spring

A spring with restoring force

$$
F_s(x)=-f(x)
$$

is nonlinear when $f(x)$ is not proportional to $x$. A common model is

$$
f(x)=kx+\alpha x^3.
$$

The spring is called hardening when $\alpha>0$ and softening when $\alpha<0$ over the physically relevant range.

### Model 5.3-B — Duffing-type oscillator

A driven damped nonlinear spring can be modeled by

$$
mx''+cx'+kx+\alpha x^3=F(t).
$$

### Proposition 5.3-C — Energy for conservative nonlinear springs

For

$$
mx''+f(x)=0,
$$

define

$$
V(x)=\int_0^x f(s)\,ds.
$$

Then

$$
E=\frac12m(x')^2+V(x)
$$

is constant.

### Model 5.3-D — Nonlinear pendulum

For pendulum angle $\theta$,

$$
\theta''+\frac{g}{\ell}\sin\theta=0.
$$

The linearization near $\theta=0$ is

$$
\theta''+\frac{g}{\ell}\theta=0.
$$

### Proposition 5.3-E — Pendulum energy threshold

The conserved energy is

$$
E=\frac12\ell^2(\theta')^2+g\ell(1-\cos\theta).
$$

The separatrix energy is $E=2g\ell$. Lower energy gives oscillation; larger energy permits rotation.

### Model 5.3-F — Catenary

A uniform flexible cable hanging under its own weight satisfies

$$
y''=a\sqrt{1+(y')^2},
\qquad a>0.
$$

With $y'(0)=0$ and $y(0)=y_0$,

$$
y(x)=\frac1a\cosh(ax)+y_0-\frac1a.
$$

### Model 5.3-G — Inverse-square rocket motion

Ignoring drag and thrust after burnout,

$$
y''=-\frac{\mu}{y^2},
$$

where $y$ is distance from the planet center and $\mu=GM$.

### Theorem 5.3-H — Escape speed

At radius $R$, the minimum speed for nonnegative total energy is

$$
v_e=\sqrt{\frac{2\mu}{R}}.
$$

### Principle 5.3-I — Variable mass

For variable mass, use momentum balance rather than automatically writing $F=ma$. The system boundary and momentum flux must be specified.

### Classroom Checkpoint — Pendulum Linearization

Which approximation converts the nonlinear pendulum equation into a linear harmonic oscillator for small angles?

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. Nonlinear restoring forces

Compare

$$
f(x)=kx,
\qquad
f(x)=kx+\alpha x^3.
$$

A hardening spring becomes increasingly stiff at large displacement. A softening model has a decreasing incremental stiffness and may be valid only over a limited range.

In [ ]:
def restoring_force_explorer(k=1.0, alpha=0.4):
    x=np.linspace(-2.5,2.5,700)
    f=k*x+alpha*x**3
    plt.plot(x,k*x,label='linear')
    plt.plot(x,f,label='nonlinear')
    plt.axhline(0,linestyle='--'); plt.axvline(0,linestyle='--')
    plt.xlabel('displacement x')
    plt.ylabel('restoring magnitude f(x)')
    plt.title('Hardening or softening spring law')
    plt.legend(); plt.show()
    print('incremental stiffness at x=2:',k+3*alpha*4)

if WIDGETS_AVAILABLE:
    interact(
        restoring_force_explorer,
        k=FloatSlider(min=0.2,max=4,step=0.2,value=1),
        alpha=FloatSlider(min=-1,max=1,step=0.05,value=0.4)
    )
else:
    restoring_force_explorer()

## 2. Conservative nonlinear spring

For

$$
x''+x+\alpha x^3=0,
$$

the potential is

$$
V(x)=\frac12x^2+\frac{\alpha}{4}x^4.
$$

When $\alpha>0$, larger amplitude produces faster oscillations. This amplitude dependence cannot occur in the ideal linear oscillator.

In [ ]:
def nonlinear_spring_explorer(alpha=0.5, x0=1.5, v0=0.0, final_time=30):
    def accel(t,x,v): return -x-alpha*x**3
    sol=solve_second_order(accel,(0,final_time),[x0,v0],points=1800,rtol=1e-10,atol=1e-12)
    x,v=sol.y
    V=0.5*x**2+alpha*x**4/4
    E=0.5*v**2+V
    plt.plot(sol.t,x)
    plt.xlabel('time'); plt.ylabel('x(t)'); plt.title('Nonlinear spring motion'); plt.show()
    plt.plot(x,v)
    plt.xlabel('x'); plt.ylabel("x'"); plt.title('Phase portrait'); plt.show()
    print('maximum energy drift =',np.max(np.abs(E-E[0])))

if WIDGETS_AVAILABLE:
    interact(
        nonlinear_spring_explorer,
        alpha=FloatSlider(min=-0.4,max=1.5,step=0.05,value=0.5),
        x0=FloatSlider(min=0.1,max=2.5,step=0.1,value=1.5),
        v0=FloatSlider(min=-3,max=3,step=0.1,value=0),
        final_time=IntSlider(min=10,max=80,step=5,value=30)
    )
else:
    nonlinear_spring_explorer()

### Amplitude-dependent period

Estimate one full period from the numerical trajectory for several initial amplitudes.

In [ ]:
def estimate_period(alpha, amplitude):
    def accel(t,x,v): return -x-alpha*x**3
    sol=solve_second_order(accel,(0,50),[amplitude,0],points=12000,rtol=1e-9,atol=1e-11)
    x=sol.y[0]
    # downward crossings of x=0; two successive downward crossings differ by a period
    idx=np.where((x[:-1]>0)&(x[1:]<=0)&(sol.y[1][1:]<0))[0]
    if len(idx)>=2:
        return sol.t[idx[1]]-sol.t[idx[0]]
    return np.nan

amps=np.linspace(0.1,2.2,25)
for alpha,label in [(0.5,'hardening'),(-0.1,'softening')]:
    periods=np.array([estimate_period(alpha,a) for a in amps])
    plt.plot(amps,periods,label=label)
plt.axhline(2*np.pi,linestyle='--',label='linear period')
plt.xlabel('initial amplitude')
plt.ylabel('estimated period')
plt.title('Nonlinear period depends on amplitude')
plt.legend(); plt.show()

## 3. Nonlinear pendulum

Tangential force balance gives

$$
m\ell\theta''=-mg\sin\theta.
$$

For small angles in radians,

$$
\sin\theta\approx\theta.
$$

The approximation is local; it deteriorates as amplitude increases.

In [ ]:
def pendulum_compare(theta0=0.5, omega0=0.0, length=1.0, final_time=20):
    g=9.81
    def accel(t,theta,omega): return -(g/length)*np.sin(theta)
    sol=solve_second_order(accel,(0,final_time),[theta0,omega0],points=1600,rtol=1e-10,atol=1e-12)
    wlin=np.sqrt(g/length)
    linear=theta0*np.cos(wlin*sol.t)+(omega0/wlin)*np.sin(wlin*sol.t)
    plt.plot(sol.t,sol.y[0],label='nonlinear')
    plt.plot(sol.t,linear,linestyle='--',label='small-angle model')
    plt.xlabel('time'); plt.ylabel('angle')
    plt.title('Nonlinear pendulum versus linearization')
    plt.legend(); plt.show()
    E=0.5*length**2*sol.y[1]**2+g*length*(1-np.cos(sol.y[0]))
    print('initial energy =',E[0])
    print('separatrix energy =',2*g*length)

if WIDGETS_AVAILABLE:
    interact(
        pendulum_compare,
        theta0=FloatSlider(min=0.05,max=3.0,step=0.05,value=0.5),
        omega0=FloatSlider(min=-8,max=8,step=0.2,value=0),
        length=FloatSlider(min=0.25,max=4,step=0.25,value=1),
        final_time=IntSlider(min=5,max=60,step=5,value=20)
    )
else:
    pendulum_compare()

### Pendulum phase portrait and separatrix

In [ ]:
g,L=9.81,1.0
th=np.linspace(-2*np.pi,2*np.pi,700)
sep=np.sqrt(np.maximum(0,2*(2*g*L-g*L*(1-np.cos(th)))/L**2))
plt.plot(th,sep,label='separatrix')
plt.plot(th,-sep)
for theta0,omega0 in [(0.7,0),(2.6,0),(0.7,6.5)]:
    def accel(t,theta,omega): return -(g/L)*np.sin(theta)
    sol=solve_second_order(accel,(0,20),[theta0,omega0],points=1600,rtol=1e-9,atol=1e-11)
    plt.plot(sol.y[0],sol.y[1],label=f'({theta0},{omega0})')
plt.xlabel('angle')
plt.ylabel('angular velocity')
plt.title('Oscillation, separatrix, and rotation')
plt.legend(); plt.show()

## 4. Shape of a hanging cable

For a uniform cable, the vertical load is proportional to arc length. The resulting equation can be written

$$
y''=a\sqrt{1+(y')^2}.
$$

The solution is a shifted and scaled hyperbolic cosine, called a **catenary**.

In [ ]:
def catenary_explorer(a=0.6, y0=0.0, span=5.0):
    x=np.linspace(-span,span,700)
    y=np.cosh(a*x)/a+y0-1/a
    parabola=y0+0.5*a*x**2
    plt.plot(x,y,label='catenary')
    plt.plot(x,parabola,linestyle='--',label='local parabolic approximation')
    plt.xlabel('horizontal position')
    plt.ylabel('height')
    plt.title('Uniform cable under its own weight')
    plt.legend(); plt.show()

if WIDGETS_AVAILABLE:
    interact(
        catenary_explorer,
        a=FloatSlider(min=0.1,max=2,step=0.1,value=0.6),
        y0=FloatSlider(min=-3,max=3,step=0.25,value=0),
        span=FloatSlider(min=1,max=10,step=0.5,value=5)
    )
else:
    catenary_explorer()

## 5. Rocket motion after burnout

For

$$
y''=-\frac{\mu}{y^2},
$$

multiplying by $y'$ and integrating gives the energy relation

$$
\frac12(y')^2-\frac{\mu}{y}=C.
$$

At launch radius $R$, escape requires

$$
\frac12v_0^2-\frac{\mu}{R}\ge0.
$$

In [ ]:
def rocket_explorer(speed_ratio=0.9, final_time=8.0):
    # nondimensional units: mu=1, R=1, escape speed=sqrt(2)
    mu,R=1.0,1.0
    ve=np.sqrt(2*mu/R)
    v0=speed_ratio*ve
    def accel(t,y,v): return -mu/y**2
    def hit_surface(t,z): return z[0]-R
    hit_surface.terminal=True
    hit_surface.direction=-1
    sol=solve_second_order(accel,(0,final_time),[R,v0],points=1500,rtol=1e-10,atol=1e-12,events=hit_surface)
    plt.plot(sol.t,sol.y[0])
    plt.axhline(R,linestyle='--',label='surface')
    plt.xlabel('time'); plt.ylabel('radius')
    plt.title('Post-burnout rocket motion')
    plt.legend(); plt.show()
    E=0.5*v0**2-mu/R
    print('escape speed =',ve)
    print('initial energy =',E)
    print('classification =','escape' if E>=0 else 'bound return')

if WIDGETS_AVAILABLE:
    interact(
        rocket_explorer,
        speed_ratio=FloatSlider(min=0.1,max=1.5,step=0.05,value=0.9),
        final_time=FloatSlider(min=1,max=30,step=1,value=8)
    )
else:
    rocket_explorer()

## Optional extension — Variable mass and a lifted chain

Newton's law in the form $F=ma$ is not automatically valid for an open system whose mass changes. One must account for momentum entering or leaving the chosen system.

For a chain being lifted from a pile, both the moving mass and its momentum change as more chain leaves the ground. This produces a nonlinear equation involving $x$, $x'$, and $x''$.

In [ ]:
# Illustrative closed-form height from an idealized constant-force chain model
# x(t)=4*sqrt(10)*t - 8*t^2 is physically meaningful only while x>=0 and x<=chain length.
t=np.linspace(0,1.5,500)
x=4*np.sqrt(10)*t-8*t**2
mask=(x>=0)&(x<=10)
plt.plot(t[mask],x[mask])
plt.xlabel('time')
plt.ylabel('lifted height')
plt.title('Idealized variable-mass chain model')
plt.show()

## Classroom Checkpoint — Exit Check

For a pendulum of length $\ell$, what is the small-angle period?

> Pause here. Let students commit to an answer before running the next cell.